In [1]:
import gc
import os

import numpy as np
import torch
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from rdkit import rdBase
from torch.utils.data import BatchSampler, DataLoader, RandomSampler, SequentialSampler

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback
from benchmarks.mlp import MLP
from dataset.fingerprint import PAC5FingerprintDataset

rdBase.DisableLog("rdApp.*")
torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
dataset = "PAC5-3"
model_path_template = f"../models/mlp_1.1M-fingerprint-[{dataset}]-fold_{{fold}}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = f"mlp_1.1M-fingerprint-[{dataset}]-fold_{{fold}}"
num_tasks = len(target_properties)
loss_weights = torch.ones(num_tasks, device=device)

W0927 23:09:26.068000 27680 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Using device: cuda


In [2]:
# -----------------------------------------------------------------------------
# 1. Training Loop
# -----------------------------------------------------------------------------
fold_results = []

for fold in range(1, k_folds + 1):
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    val_path = model_dataset_dir + f"/{dataset}/fold_{fold}_val.csv"

    train_dataset = PAC5FingerprintDataset(train_path, properties=target_properties)
    val_dataset = PAC5FingerprintDataset(val_path, properties=target_properties)

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.y.numpy())

    train_dataset.y = torch.tensor(
        scaler.transform(train_dataset.y.numpy()), dtype=torch.float32
    )
    val_dataset.y = torch.tensor(
        scaler.transform(val_dataset.y.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=None,
        sampler=BatchSampler(
            RandomSampler(train_dataset), batch_size=256, drop_last=False
        ),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=None,
        sampler=BatchSampler(
            SequentialSampler(val_dataset), batch_size=256, drop_last=False
        ),
    )

    core_model = MLP(input_dim=2048, hidden_dim=468, num_tasks=num_tasks)
    lightning_model = LightningModule(core_model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=10,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.5f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = LightningModule.load_from_checkpoint(
        best_model_path,
        model=core_model,
        property_names=target_properties,
        scaler=scaler,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    del train_dataset
    del val_dataset
    del train_loader
    del val_loader
    del core_model
    del lightning_model
    del trainer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Final k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 1/5 ]======


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
D:\Coding\pac-gcnn\.venv\L

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ MLP        │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss    │      0 │ train │     0 │
│ 2 │ train_rmsd │ ModuleDict │      0 │ train │     0 │
│ 3 │ val_rmsd   │ ModuleDict │      0 │ train │     0 │
│ 4 │ test_rmsd  │ ModuleDict │      0 │ train │     0 │
│ 5 │ test_mae   │ ModuleDict │      0 │ train │     0 │
│ 6 │ test_r2    │ ModuleDict │      0 │ train │     0 │
│ 7 │ test_max   │ ModuleDict │      0 │ train │     0 │
│ 8 │ test_min   │ ModuleDict │      0 │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.395                                                                      
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


...

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ MLP        │  1.1 M │ train │     0 │
│ 1 │ criterion  │ MSELoss    │      0 │ train │     0 │
│ 2 │ train_rmsd │ ModuleDict │      0 │ train │     0 │
│ 3 │ val_rmsd   │ ModuleDict │      0 │ train │     0 │
│ 4 │ test_rmsd  │ ModuleDict │      0 │ train │     0 │
│ 5 │ test_mae   │ ModuleDict │      0 │ train │     0 │
│ 6 │ test_r2    │ ModuleDict │      0 │ train │     0 │
│ 7 │ test_max   │ ModuleDict │      0 │ train │     0 │
│ 8 │ test_min   │ ModuleDict │      0 │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4.395                                                                      
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Epoch 001 | Scaled Train Loss: N/A | Scaled Val Loss: 0.12394 | xtb_homo Val RMSD: 0.13785 | xtb_lumo Val RMSD: 0.16341 | xtb_hl_gap Val RMSD: 0.18161 | pm7_homo Val RMSD: 0.14998 | pm7_lumo Val RMSD: 0.13471 | pm7_hl_gap Val RMSD: 0.18257 | pm7_hof Val RMSD: 107.61268
Epoch 002 | Scaled Train Loss: 0.20192 | Scaled Val Loss: 0.09330 | xtb_homo Val RMSD: 0.11832 | xtb_lumo Val RMSD: 0.13712 | xtb_hl_gap Val RMSD: 0.15685 | pm7_homo Val RMSD: 0.13107 | pm7_lumo Val RMSD: 0.11613 | pm7_hl_gap Val RMSD: 0.16124 | pm7_hof Val RMSD: 96.99566
Epoch 003 | Scaled Train Loss: 0.10268 | Scaled Val Loss: 0.08042 | xtb_homo Val RMSD: 0.10830 | xtb_lumo Val RMSD: 0.12551 | xtb_hl_gap Val RMSD: 0.14511 | pm7_homo Val RMSD: 0.12191 | pm7_lumo Val RMSD: 0.10711 | pm7_hl_gap Val RMSD: 0.15265 | pm7_hof Val RMSD: 91.46465
Epoch 004 | Scaled Train Loss: 0.07805 | Scaled Val Loss: 0.07269 | xtb_homo Val RMSD: 0.10316 | xtb_lumo Val RMSD: 0.11836 | xtb_hl_gap Val RMSD: 0.13814 | pm7_homo Val RMSD: 0.11711 

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100 | Scaled Train Loss: 0.00594 | Scaled Val Loss: 0.04858 | xtb_homo Val RMSD: 0.08450 | xtb_lumo Val RMSD: 0.09489 | xtb_hl_gap Val RMSD: 0.11267 | pm7_homo Val RMSD: 0.09741 | pm7_lumo Val RMSD: 0.08483 | pm7_hl_gap Val RMSD: 0.11968 | pm7_hof Val RMSD: 69.56913
Best scaled validation loss for fold 5: 0.04546

======[ Final k-fold results ]======
Scaled Loss per fold: [0.0455, 0.047, 0.0456, 0.0464, 0.0455]
Average Scaled Loss: 0.0460 ± 0.0006


In [3]:
# -----------------------------------------------------------------------------
# 2. Evaluation Loop
# -----------------------------------------------------------------------------

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {
        property_name: {metric: [] for metric in metric_keys}
        for property_name in target_properties
    },
    "test": {
        property_name: {metric: [] for metric in metric_keys}
        for property_name in target_properties
    },
}

for fold in range(1, k_folds + 1):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    test_path = model_dataset_dir + f"/{dataset}/fold_{fold}_test.csv"

    train_dataset = PAC5FingerprintDataset(train_path, properties=target_properties)
    test_dataset = PAC5FingerprintDataset(test_path, properties=target_properties)

    scaler = MultiTargetScaler()
    scaler.fit(train_dataset.y.numpy())

    train_dataset.y = torch.tensor(
        scaler.transform(train_dataset.y.numpy()), dtype=torch.float32
    )
    test_dataset.y = torch.tensor(
        scaler.transform(test_dataset.y.numpy()), dtype=torch.float32
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=None,
        sampler=BatchSampler(
            SequentialSampler(train_dataset), batch_size=256, drop_last=False
        ),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=None,
        sampler=BatchSampler(
            SequentialSampler(test_dataset), batch_size=256, drop_last=False
        ),
    )

    core_model = MLP(input_dim=2048, hidden_dim=468, num_tasks=num_tasks)
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    core_model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = LightningModule(core_model, target_properties, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("\n======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        for metric in metric_keys:
            all_results["train"][prop][metric].append(
                train_results[f"test_{metric}_{prop}"]
            )
            all_results["test"][prop][metric].append(
                test_results[f"test_{metric}_{prop}"]
            )

    del train_dataset
    del test_dataset
    del train_loader
    del test_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Cross-validation results (mean ± std) ]======")

for property_name in target_properties:
    print(f"\n======[ {property_name} ]======")

    train_statistics = {
        metric: (
            np.mean(all_results["train"][property_name][metric]),
            np.std(all_results["train"][property_name][metric], ddof=1),
        )
        for metric in metric_keys
    }

    print(
        f" + Train -> "
        f"MAE: {train_statistics['mae'][0]:.4f} ± {train_statistics['mae'][1]:.4f}\t"
        f"RMSD: {train_statistics['rmsd'][0]:.4f} ± {train_statistics['rmsd'][1]:.4f}\t"
        f"Max: {train_statistics['max'][0]:.4f} ± {train_statistics['max'][1]:.4f}\t"
        f"Min: {train_statistics['min'][0]:.4f} ± {train_statistics['min'][1]:.4f}\t"
        f"R^2: {train_statistics['r2'][0]:.4f} ± {train_statistics['r2'][1]:.4f}"
    )

    test_statistics = {
        metric: (
            np.mean(all_results["test"][property_name][metric]),
            np.std(all_results["test"][property_name][metric], ddof=1),
        )
        for metric in metric_keys
    }

    print(
        f" + Test -> "
        f"MAE: {test_statistics['mae'][0]:.4f} ± {test_statistics['mae'][1]:.4f}\t"
        f"RMSD: {test_statistics['rmsd'][0]:.4f} ± {test_statistics['rmsd'][1]:.4f}\t"
        f"Max: {test_statistics['max'][0]:.4f} ± {test_statistics['max'][1]:.4f}\t"
        f"Min: {test_statistics['min'][0]:.4f} ± {test_statistics['min'][1]:.4f}\t"
        f"R^2: {test_statistics['r2'][0]:.4f} ± {test_statistics['r2'][1]:.4f}"
    )


======[ Evaluating fold 1/5 ]======


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.



======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0262	RMSD: 0.0357	Max: 0.6547	Min: 0.0000	R^2: 0.9932
 + Test -> MAE: 0.0495	RMSD: 0.0807	Max: 1.9123	Min: 0.0000	R^2: 0.9651
======[ xtb_lumo ]======
 + Train -> MAE: 0.0301	RMSD: 0.0404	Max: 0.6129	Min: 0.0000	R^2: 0.9943
 + Test -> MAE: 0.0580	RMSD: 0.0924	Max: 1.5878	Min: 0.0000	R^2: 0.9697
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0324	RMSD: 0.0445	Max: 1.0774	Min: 0.0000	R^2: 0.9884
 + Test -> MAE: 0.0671	RMSD: 0.1093	Max: 2.1532	Min: 0.0000	R^2: 0.9297
======[ pm7_homo ]======
 + Train -> MAE: 0.0332	RMSD: 0.0450	Max: 0.6778	Min: 0.0000	R^2: 0.9922
 + Test -> MAE: 0.0605	RMSD: 0.0932	Max: 2.4019	Min: 0.0000	R^2: 0.9663
======[ pm7_lumo ]======
 + Train -> MAE: 0.0278	RMSD: 0.0374	Max: 0.7061	Min: 0.0000	R^2: 0.9886
 + Test -> MAE: 0.0533	RMSD: 0.0828	Max: 2.7522	Min: 0.0000	R^2: 0.9437
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0370	RMSD: 0.0514	Max: 1.3453	Min: 0.0000	R^2: 0.9872
 + Test -> 

...


======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0279	RMSD: 0.0380	Max: 0.5727	Min: 0.0000	R^2: 0.9923
 + Test -> MAE: 0.0500	RMSD: 0.0810	Max: 1.2727	Min: 0.0000	R^2: 0.9648
======[ xtb_lumo ]======
 + Train -> MAE: 0.0324	RMSD: 0.0434	Max: 0.9017	Min: 0.0000	R^2: 0.9934
 + Test -> MAE: 0.0585	RMSD: 0.0927	Max: 2.1198	Min: 0.0000	R^2: 0.9698
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0349	RMSD: 0.0478	Max: 0.9513	Min: 0.0000	R^2: 0.9866
 + Test -> MAE: 0.0677	RMSD: 0.1093	Max: 1.6608	Min: 0.0000	R^2: 0.9301
======[ pm7_homo ]======
 + Train -> MAE: 0.0352	RMSD: 0.0476	Max: 0.6891	Min: 0.0000	R^2: 0.9912
 + Test -> MAE: 0.0611	RMSD: 0.0934	Max: 1.5123	Min: 0.0000	R^2: 0.9663
======[ pm7_lumo ]======
 + Train -> MAE: 0.0297	RMSD: 0.0397	Max: 0.6620	Min: 0.0000	R^2: 0.9871
 + Test -> MAE: 0.0536	RMSD: 0.0817	Max: 1.3414	Min: 0.0000	R^2: 0.9456
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0407	RMSD: 0.0557	Max: 1.2407	Min: 0.0000	R^2: 0.9849
 + Test -> 

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Train Set Evaluation ]======


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



======[ Test Set Evaluation ]======
======[ xtb_homo ]======
 + Train -> MAE: 0.0277	RMSD: 0.0378	Max: 0.6911	Min: 0.0000	R^2: 0.9923
 + Test -> MAE: 0.0503	RMSD: 0.0818	Max: 1.9588	Min: 0.0000	R^2: 0.9643
======[ xtb_lumo ]======
 + Train -> MAE: 0.0317	RMSD: 0.0427	Max: 0.7551	Min: 0.0000	R^2: 0.9936
 + Test -> MAE: 0.0586	RMSD: 0.0936	Max: 1.4244	Min: 0.0000	R^2: 0.9693
======[ xtb_hl_gap ]======
 + Train -> MAE: 0.0339	RMSD: 0.0469	Max: 1.4188	Min: 0.0000	R^2: 0.9872
 + Test -> MAE: 0.0680	RMSD: 0.1109	Max: 2.2862	Min: 0.0000	R^2: 0.9277
======[ pm7_homo ]======
 + Train -> MAE: 0.0355	RMSD: 0.0480	Max: 0.8840	Min: 0.0000	R^2: 0.9911
 + Test -> MAE: 0.0612	RMSD: 0.0937	Max: 2.0773	Min: 0.0000	R^2: 0.9661
======[ pm7_lumo ]======
 + Train -> MAE: 0.0292	RMSD: 0.0394	Max: 0.9262	Min: 0.0000	R^2: 0.9874
 + Test -> MAE: 0.0539	RMSD: 0.0830	Max: 2.1570	Min: 0.0000	R^2: 0.9439
======[ pm7_hl_gap ]======
 + Train -> MAE: 0.0398	RMSD: 0.0553	Max: 1.8106	Min: 0.0000	R^2: 0.9852
 + Test -> 